# Ensemble Learning for Myocardial Perfusion PET Polar-Map Classification

## Overview
This notebook applies four ensemble rules—Majority Voting, Sum, Max, and Median—to manually specified Top-3 and Top-5 transfer-learning model pools. The eight configurations are evaluated run-matched across repeated transfer-learning runs.

## Inputs
- A TL workbook in either supported format: the original one-sheet workbook with `model_name`, `predicts`, and `tag`; or the current run-level workbook with `Run_results`, `test_probabilities`, and `seed`.
- `test_labels.txt`: one binary test-set label per line in the identical order as the saved comma-separated test probabilities.

## Outputs
- A printed median/IQR summary for the eight configurations.
- `results_EL.xlsx`: one `Run_results` sheet with 800 raw rows (8 configurations × 100 run-matched runs). Probability-producing rules retain comma-separated `predicts`; Majority Voting retains comma-separated binary predictions and has no probability/AUC value.


In [ ]:
# Cell 1: Imports and configuration
import pandas as pd
import numpy as np
import os
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, roc_auc_score
)

# ── User configuration ──────────────────────────────────────────────────────
DATA_DIR     = '../data'              # Directory containing input files
EXCEL_FILE   = 'results_TL.xlsx'     # TL workbook with 11 × 100 runs
LABELS_FILE  = 'test_labels.txt'      # True test-set labels in saved-score order
OUTPUT_FILE  = 'results_EL.xlsx'  # Raw 8 × 100 ensemble-run workbook
NUM_RUNS     = 100
OVERWRITE_OUTPUT = False               # Change only when intentionally replacing output
# ─────────────────────────────────────────────────────────────────────────────

excel_path = os.path.join(DATA_DIR, EXCEL_FILE)
labels_path = os.path.join(DATA_DIR, LABELS_FILE)

# Support both the original one-sheet workbook and the current run-level form.
input_sheets = pd.read_excel(excel_path, sheet_name=None)
input_sheet = 'Run_results' if 'Run_results' in input_sheets else next(iter(input_sheets))
df_models = input_sheets[input_sheet].copy()
PREDICTION_COLUMN = 'predicts' if 'predicts' in df_models.columns else 'test_probabilities'
RUN_COLUMN = 'tag' if 'tag' in df_models.columns else 'seed'
if PREDICTION_COLUMN not in df_models.columns or RUN_COLUMN not in df_models.columns:
    raise ValueError('TL workbook must contain model_name plus prediction and run-identifier columns.')

with open(labels_path, 'r') as f:
    true_labels = np.array([int(line.strip()) for line in f if line.strip()])
N_TEST = len(true_labels)

print(f'Loaded {len(df_models)} model-run records from sheet: {input_sheet}')
print(f'Using prediction column: {PREDICTION_COLUMN}; run identifier: {RUN_COLUMN}')
print(f'Test set: {N_TEST} images | {np.sum(true_labels)} class 1, {N_TEST - np.sum(true_labels)} class 0')
print(f'Models found: {sorted(df_models["model_name"].astype(str).unique())}')


In [ ]:
# Cell 2: Define development-ranked model pools
# Enter pools determined during development before test-set ensemble evaluation.
TOP3_MODELS = [
    'InceptionResNetV2',
    'DenseNet201',
    'Xception',
]
TOP5_MODELS = [
    'InceptionResNetV2',
    'DenseNet201',
    'Xception',
    'MobileNetV2',
    'VGG19',
]

if TOP3_MODELS != TOP5_MODELS[:3] or len(set(TOP5_MODELS)) != 5:
    raise ValueError('TOP3_MODELS must be the first three entries of the unique TOP5_MODELS list.')
POOLS = {'Top-3': TOP3_MODELS, 'Top-5': TOP5_MODELS}


In [ ]:
# Cell 3: Helper functions

def parse_predictions(pred_string):
    """Parse a comma- or space-separated string of probabilities into an array."""
    if isinstance(pred_string, (list, np.ndarray)):
        values = np.asarray(pred_string, dtype=float)
    else:
        cleaned = str(pred_string).strip('[](){} ')
        separator = ',' if ',' in cleaned else None
        values = np.array([float(x) for x in cleaned.split(separator) if x.strip()], dtype=float)
    if len(values) != N_TEST:
        raise ValueError(f'Expected {N_TEST} test probabilities, found {len(values)}.')
    return values


def model_run_rows(model_name):
    rows = df_models.loc[df_models['model_name'].astype(str) == str(model_name)].copy()
    if len(rows) < NUM_RUNS:
        raise ValueError(f"Model '{model_name}' has {len(rows)} runs; {NUM_RUNS} are required.")
    return rows.sort_values(RUN_COLUMN, kind='stable').reset_index(drop=True)


def get_run_predictions(model_names, run_idx):
    """Return run-matched class-1 test probabilities with shape (K, N_TEST)."""
    probabilities = []
    for name in model_names:
        rows = model_run_rows(name)
        probabilities.append(parse_predictions(rows.loc[run_idx, PREDICTION_COLUMN]))
    return np.vstack(probabilities)


def vector_to_text(values):
    return ','.join(format(float(value), '.17g') for value in np.asarray(values))


def compute_metrics(y_true, y_pred, y_prob=None):
    """Return metrics and confusion counts for one 46-patient test evaluation."""
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        'ACC': accuracy_score(y_true, y_pred),
        'PRE': precision_score(y_true, y_pred, zero_division=0),
        'SEN': recall_score(y_true, y_pred, zero_division=0),
        'SPE': tn / (tn + fp) if (tn + fp) else np.nan,
        'F1S': f1_score(y_true, y_pred, zero_division=0),
        'AUC': roc_auc_score(y_true, y_prob) if y_prob is not None else np.nan,
        'TN': int(tn), 'FP': int(fp), 'FN': int(fn), 'TP': int(tp),
    }


In [ ]:
# Cell 4: Retained ensemble-rule implementations

def majority_voting(probs):
    """Strict-majority binary voting at the fixed 0.50 constituent threshold."""
    votes = (probs >= 0.5).astype(int)
    return (votes.sum(axis=0) > probs.shape[0] / 2).astype(int), None


def sum_rule(probs):
    """Arithmetic mean of constituent class-1 probabilities; threshold = 0.50."""
    y_prob = probs.mean(axis=0)
    return (y_prob >= 0.5).astype(int), y_prob


def max_rule(probs):
    """Maximum constituent class-1 probability; threshold = 0.50."""
    y_prob = probs.max(axis=0)
    return (y_prob >= 0.5).astype(int), y_prob


def median_rule(probs):
    """Median constituent class-1 probability; threshold = 0.50."""
    y_prob = np.median(probs, axis=0)
    return (y_prob >= 0.5).astype(int), y_prob

RULES = {
    'Majority Voting': majority_voting,
    'Sum Rule': sum_rule,
    'Max Rule': max_rule,
    'Median Rule': median_rule,
}


In [ ]:
# Cell 5: Run the 8 retained configurations (4 rules × 2 pools)
all_results = {}
raw_rows = []

for pool_name, models in POOLS.items():
    for run_idx in range(NUM_RUNS):
        probs = get_run_predictions(models, run_idx)
        run_identifier = int(run_idx + 1)
        for rule_name, rule_fn in RULES.items():
            y_pred, y_prob = rule_fn(probs)
            metrics = compute_metrics(true_labels, y_pred, y_prob)
            key = (pool_name, rule_name)
            all_results.setdefault(key, []).append(metrics)
            probability_text = '' if y_prob is None else vector_to_text(y_prob)
            binary_text = vector_to_text(y_pred)
            raw_rows.append({
                'model_name': f'{pool_name} {rule_name}',
                'pool': pool_name,
                'rule': rule_name,
                'tag': run_identifier,
                'threshold': 0.5,
                'test_accuracy': metrics['ACC'],
                'test_precision': metrics['PRE'],
                'test_recall': metrics['SEN'],
                'test_sensitivity': metrics['SEN'],
                'test_specificity': metrics['SPE'],
                'test_f1score': metrics['F1S'],
                'test_f1': metrics['F1S'],
                'test_auc': metrics['AUC'],
                'test_tn': metrics['TN'],
                'test_fp': metrics['FP'],
                'test_fn': metrics['FN'],
                'test_tp': metrics['TP'],
                'test_confusion_matrix': f"[[{metrics['TN']}, {metrics['FP']}], [{metrics['FN']}, {metrics['TP']}]]",
                'test_probabilities': probability_text,
                'test_binary_predictions': binary_text,
                # Legacy alias retained for existing comma-separated readers.
                'predicts': probability_text,
                'binary_predictions': binary_text,
            })

print(f'Completed {len(all_results)} configurations × {NUM_RUNS} run-matched evaluations.')
if len(raw_rows) != 8 * NUM_RUNS:
    raise RuntimeError(f'Expected {8 * NUM_RUNS} raw rows, found {len(raw_rows)}.')


In [ ]:
# Cell 6: Summarise results — median and IQR across 100 runs
METRIC_COLS = ['ACC', 'PRE', 'SEN', 'SPE', 'F1S', 'AUC']
RULE_ORDER = ['Majority Voting', 'Sum Rule', 'Max Rule', 'Median Rule']

summary_rows = []
for pool_name in ['Top-3', 'Top-5']:
    for rule_name in RULE_ORDER:
        run_metrics = pd.DataFrame(all_results[(pool_name, rule_name)])
        row = {'Pool': pool_name, 'Rule': rule_name}
        for col in METRIC_COLS:
            vals = run_metrics[col].dropna()
            row[f'{col}_Median'] = np.median(vals) if len(vals) else np.nan
            row[f'{col}_IQR'] = np.percentile(vals, 75) - np.percentile(vals, 25) if len(vals) else np.nan
        summary_rows.append(row)
summary_df = pd.DataFrame(summary_rows)

pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 180)
print(summary_df.to_string(index=False))


In [ ]:
# Cell 7: Write the raw 800-row ensemble workbook
output_path = os.path.abspath(OUTPUT_FILE)
if os.path.exists(output_path) and not OVERWRITE_OUTPUT:
    raise FileExistsError(f'Output already exists: {output_path}. Set OVERWRITE_OUTPUT=True only to replace it intentionally.')

raw_results_df = pd.DataFrame(raw_rows)
with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
    raw_results_df.to_excel(writer, sheet_name='Run_results', index=False)

print(f'Wrote {len(raw_results_df)} raw ensemble rows to {output_path}')
